# Experiment 3 Pose Roads

Reproducibility notebook for the CNN experiments in *Are You Thinking What I'm Thinking?*.

- The first code cell installs packages from `requirements.txt` only if they are missing.
- Expected image folders are documented in `data/README.md`.
- Set `CNN_DATA_ROOT`, `POSE_DATA_ROOT`, or `ROAD_DATA_ROOT` as environment variables if your images live elsewhere.
- The notebook uses pretrained ImageNet weights and extracts pre-final-layer activations.
- Outputs have been cleared from the repository version; run cells top-to-bottom to reproduce results.


In [ ]:
# Install dependencies if needed, then set repository paths and seeds.
# Standard library only until packages from requirements.txt are present.
from pathlib import Path
import os
import sys
import subprocess
import random

def find_repo_root(start=Path.cwd()):
    for candidate in [start, *start.parents]:
        if (candidate / "data").exists() and (candidate / "cnns").exists():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the cloned repository.")

REPO_ROOT = find_repo_root()

REQUIRED_MODULES = (
    "numpy", "pandas", "matplotlib", "seaborn", "scipy", "sklearn",
    "PIL", "openpyxl", "torch", "torchvision", "transformers",
    "accelerate", "huggingface_hub",
)

def dependencies_present():
    import importlib
    for name in REQUIRED_MODULES:
        try:
            importlib.import_module(name)
        except ImportError:
            return False
    return True

req_file = REPO_ROOT / "requirements.txt"
if dependencies_present():
    print("Dependencies already installed; skipping pip.")
else:
    print(f"Installing packages from {req_file} (first run can take several minutes)...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", str(req_file)])
    print("Dependencies ready.")

import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

CNN_DATA_ROOT = Path(os.environ.get("CNN_DATA_ROOT", REPO_ROOT / "data" / "cnn_images"))
POSE_DATA_ROOT = Path(os.environ.get("POSE_DATA_ROOT", CNN_DATA_ROOT / "Pose"))
ROAD_DATA_ROOT = Path(os.environ.get("ROAD_DATA_ROOT", CNN_DATA_ROOT / "Roads"))
RESULTS_ROOT = REPO_ROOT / "results" / "cnn"
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Repository root: {REPO_ROOT}")
print(f"CNN data root:   {CNN_DATA_ROOT}")


# Experiment 3: Sub-Concepts and Distribution Shift

Two sub-experiments. First: do models represent sub-concepts within a class (sleeping vs standing cats)? Second: is KL divergence sensitive to distribution shift within a class (Indian vs Turkish roads)?

## Sub-Experiment 3a: Cat Pose — Sleeping vs Standing

We ask whether the models' pre-final layer activations can separate two semantically distinct sub-categories of the cat class. If pose is encoded as a meaningful internal dimension, sleeping and standing cats should form separable clusters.

### Setup and Feature Extraction

In [ ]:
# ============================================================
# CELL 1 — Imports & Setup
# ============================================================
import os
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.decomposition import PCA
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


In [ ]:
# ============================================================
# CELL 2 — Image Loader
# ============================================================
def load_images_from_folder(folder_path, label):
    """Returns list of (PIL image, label, filename)"""
    supported = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
    entries = []
    for fname in sorted(os.listdir(folder_path)):
        if fname.lower().endswith(supported):
            img = Image.open(os.path.join(folder_path, fname)).convert('RGB')
            entries.append((img, label, fname))
    return entries

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225]),
])

# Load both classes
sleeping_data = load_images_from_folder(str(POSE_DATA_ROOT / "Sleeping"), label=0)
standing_data = load_images_from_folder(str(POSE_DATA_ROOT / "Standing"), label=1)
all_data      = sleeping_data + standing_data

labels    = np.array([d[1] for d in all_data])
filenames = [d[2] for d in all_data]

print(f"Sleeping: {len(sleeping_data)} images")
print(f"Standing: {len(standing_data)} images")
print(f"Total   : {len(all_data)} images")


In [ ]:
# ============================================================
# CELL 3 — Feature Extractor (pre-final layer)  [FIXED]
# ============================================================
import torch.nn.functional as F

class MobileNetV2Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT)
        self.features = base.features   # outputs (B, 1280, 7, 7)

    def forward(self, x):
        x = self.features(x)
        x = F.adaptive_avg_pool2d(x, (1, 1))  # → (B, 1280, 1, 1)
        x = torch.flatten(x, 1)               # → (B, 1280)
        return x

class ResNet50Extractor(nn.Module):
    def __init__(self):
        super().__init__()
        base = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        # Everything except the final FC layer
        self.backbone = nn.Sequential(*list(base.children())[:-1])  # → (B, 2048, 1, 1)

    def forward(self, x):
        x = self.backbone(x)
        x = torch.flatten(x, 1)   # → (B, 2048)
        return x

mob_backbone = MobileNetV2Extractor().eval().to(device)
res_backbone = ResNet50Extractor().eval().to(device)
print("Models loaded ✓")


In [ ]:
# ============================================================
# CELL 4 — Extract Features & Run PCA
# ============================================================
print("Extracting MobileNetV2 features …")
mob_feats = extract_features(mob_backbone, all_data)

print("Extracting ResNet50 features …")
res_feats = extract_features(res_backbone, all_data)

# PCA → 3D
pca3 = PCA(n_components=3, random_state=42)
mob_pca = pca3.fit_transform(mob_feats)
print(f"MobileNetV2 explained variance (3 PCs): "
      f"{pca3.explained_variance_ratio_.sum()*100:.1f}%")

pca3b = PCA(n_components=3, random_state=42)
res_pca = pca3b.fit_transform(res_feats)
print(f"ResNet50 explained variance (3 PCs): "
      f"{pca3b.explained_variance_ratio_.sum()*100:.1f}%")


### KL Divergence — Sleeping vs Standing Baseline

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
from scipy.stats import entropy
import matplotlib.pyplot as plt

# ── CONFIG ────────────────────────────────────────────────────────────
DATA_DIR  = str(POSE_DATA_ROOT)
DEVICE    = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_BASE    = 100
N_TEST    = 100

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp")

# ── TRANSFORM ─────────────────────────────────────────────────────────
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225])
])

def load_image(path):
    return transform(Image.open(path).convert("RGB"))

def get_image_paths(folder):
    return sorted([
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(image_extensions)
    ])

# ── MODELS ────────────────────────────────────────────────────────────
# ResNet-50
print("Loading ResNet-50 ...")
resnet   = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(DEVICE)
resnet.eval()
rn_model = nn.Sequential(*list(resnet.children())[:-1])
rn_model.eval().to(DEVICE)

def rn_get_activation(img_tensor):
    with torch.no_grad():
        feat = rn_model(img_tensor).view(-1)
        feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)
        return feat.cpu().numpy()

# MobileNetV2
print("Loading MobileNetV2 ...")
mobilenet_full = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT).to(DEVICE)
mobilenet_full.eval()

class MobileNetFeatures(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.features = model.features
        self.pool     = nn.AdaptiveAvgPool2d((1, 1))
    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        return x.view(x.size(0), -1)

mn_model = MobileNetFeatures(mobilenet_full)
mn_model.eval().to(DEVICE)

def mn_get_activation(img_tensor):
    with torch.no_grad():
        feat = mn_model(img_tensor).view(-1)
        feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)
        return feat.cpu().numpy()

# ── STEP 1: BUILD SLEEPING BASELINE ───────────────────────────────────
sleeping_paths  = get_image_paths(os.path.join(DATA_DIR, "Sleeping"))
standing_paths  = get_image_paths(os.path.join(DATA_DIR, "Standing"))

def build_baseline(get_act, paths, n=N_BASE):
    feats = []
    for path in paths[:n]:
        img  = load_image(path).unsqueeze(0).to(DEVICE)
        feat = get_act(img)
        feats.append(feat)
    mean_vec = np.stack(feats).mean(axis=0)
    return mean_vec / (mean_vec.sum() + 1e-8)

print("\nBuilding baselines ...")
rn_baseline = build_baseline(rn_get_activation, sleeping_paths)
mn_baseline = build_baseline(mn_get_activation, sleeping_paths)

# ── STEP 2: COMPUTE KL FOR SLEEPING AND STANDING ──────────────────────
def compute_kl(get_act, baseline, paths, n=N_TEST):
    kl_vals = []
    for path in paths[:n]:
        img  = load_image(path).unsqueeze(0).to(DEVICE)
        feat = get_act(img)
        feat = feat / (feat.sum() + 1e-8)
        kl   = entropy(feat + 1e-8, baseline + 1e-8)
        kl_vals.append(kl)
    return kl_vals

print("\nComputing ResNet-50 KL ...")
rn_sleeping_kl = compute_kl(rn_get_activation, rn_baseline,
                             sleeping_paths[N_BASE:N_BASE + N_TEST])
rn_standing_kl = compute_kl(rn_get_activation, rn_baseline,
                             standing_paths[:N_TEST])

print("Computing MobileNetV2 KL ...")
mn_sleeping_kl = compute_kl(mn_get_activation, mn_baseline,
                             sleeping_paths[N_BASE:N_BASE + N_TEST])
mn_standing_kl = compute_kl(mn_get_activation, mn_baseline,
                             standing_paths[:N_TEST])

# ── STEP 3: PRINT RESULTS ─────────────────────────────────────────────
print("\n==============================")
print("FINAL RESULTS")
print("==============================")
print(f"ResNet-50   — Sleeping vs Sleeping baseline: {np.mean(rn_sleeping_kl):.6f}")
print(f"ResNet-50   — Standing vs Sleeping baseline: {np.mean(rn_standing_kl):.6f}")
print(f"MobileNetV2 — Sleeping vs Sleeping baseline: {np.mean(mn_sleeping_kl):.6f}")
print(f"MobileNetV2 — Standing vs Sleeping baseline: {np.mean(mn_standing_kl):.6f}")

# ── STEP 4: DISTRIBUTION SHIFT PLOTS ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150)

for ax, sleeping_kl, standing_kl, model_name in zip(
    axes,
    [rn_sleeping_kl, mn_sleeping_kl],
    [rn_standing_kl, mn_standing_kl],
    ["ResNet-50", "MobileNetV2"]
):
    ax.hist(sleeping_kl, bins=20, alpha=0.6,
            color='steelblue', label=f"Sleeping (avg={np.mean(sleeping_kl):.4f})")
    ax.hist(standing_kl, bins=20, alpha=0.6,
            color='crimson',   label=f"Standing (avg={np.mean(standing_kl):.4f})")

    ax.axvline(np.mean(sleeping_kl), color='steelblue',
               linestyle='--', linewidth=1.5)
    ax.axvline(np.mean(standing_kl), color='crimson',
               linestyle='--', linewidth=1.5)

    ax.text(np.mean(sleeping_kl), ax.get_ylim()[1] * 0.92,
            f"{np.mean(sleeping_kl):.4f}",
            color='steelblue', ha='center', fontsize=9, fontweight='bold')
    ax.text(np.mean(standing_kl), ax.get_ylim()[1] * 0.82,
            f"{np.mean(standing_kl):.4f}",
            color='crimson', ha='center', fontsize=9, fontweight='bold')

    ax.set_title(model_name, fontsize=11, fontweight='bold')
    ax.set_xlabel("KL Divergence vs Sleeping Baseline", fontsize=9)
    ax.set_ylabel("Frequency", fontsize=9)
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_facecolor('#f9f9f9')
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)

plt.suptitle("Distribution Shift — Sleeping vs Standing\n(baseline = first 100 Sleeping images)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("distribution_shift_pose.png", dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── STEP 3: PRINT RESULTS ─────────────────────────────────────────────
print("\n==============================")
print("FINAL RESULTS")
print("==============================")
print(f"ResNet-50   — Sleeping vs Sleeping baseline: {np.mean(rn_sleeping_kl):.6f}")
print(f"ResNet-50   — Standing vs Sleeping baseline: {np.mean(rn_standing_kl):.6f}")
print(f"MobileNetV2 — Sleeping vs Sleeping baseline: {np.mean(mn_sleeping_kl):.6f}")
print(f"MobileNetV2 — Standing vs Sleeping baseline: {np.mean(mn_standing_kl):.6f}")

# ── STEP 4: DISTRIBUTION SHIFT PLOTS ─────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=150)

for ax, sleeping_kl, standing_kl, model_name in zip(
    axes,
    [rn_sleeping_kl, mn_sleeping_kl],
    [rn_standing_kl, mn_standing_kl],
    ["ResNet-50", "MobileNetV2"]
):
    ax.hist(sleeping_kl, bins=20, alpha=0.6,
            color='#440154', label=f"Sleeping (avg={np.mean(sleeping_kl):.4f})")
    ax.hist(standing_kl, bins=20, alpha=0.6,
            color="#f98e09",   label=f"Standing (avg={np.mean(standing_kl):.4f})")

    ax.axvline(np.mean(sleeping_kl), color='#440154',
               linestyle='--', linewidth=1.5)
    ax.axvline(np.mean(standing_kl), color="#f98e09",
               linestyle='--', linewidth=1.5)

    ax.text(np.mean(sleeping_kl), ax.get_ylim()[1] * 0.92,
            f"{np.mean(sleeping_kl):.4f}",
            color='#440154', ha='center', fontsize=9, fontweight='bold')
    ax.text(np.mean(standing_kl), ax.get_ylim()[1] * 0.82,
            f"{np.mean(standing_kl):.4f}",
            color="#f98e09", ha='center', fontsize=9, fontweight='bold')

    ax.set_title(model_name, fontsize=11, fontweight='bold')
    ax.set_xlabel("KL Divergence vs Sleeping Baseline", fontsize=9)
    ax.set_ylabel("Frequency", fontsize=9)
    ax.legend(fontsize=8)
    ax.tick_params(labelsize=7)
    ax.set_facecolor('#f9f9f9')
    for spine in ax.spines.values():
        spine.set_linewidth(0.4)

plt.suptitle("Distribution Shift — Sleeping vs Standing\n(baseline = first 100 Sleeping images)",
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig("distribution_shift_pose.png", dpi=150, bbox_inches='tight')
plt.show()


## Sub-Experiment 3b: Road Images — India vs Turkey

We build a baseline from Indian road images and compute KL divergence for held-out Indian images and Turkish road images against this baseline. A higher divergence for Turkish images indicates that distribution shift is detectable via activation distributions without labels.

### ResNet50 — India Baseline and KL Distributions

In [ ]:
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
from scipy.stats import entropy
import matplotlib.pyplot as plt

############################################################
# CONFIG
############################################################

base_path = str(ROAD_DATA_ROOT)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

num_images_avg = 100
num_images_test = 100

image_extensions = (".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tiff", ".webp")


############################################################
# LOAD RESNET50
############################################################

resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT).to(device)
resnet.eval()

# Remove final FC layer → Penultimate activations
model = nn.Sequential(*list(resnet.children())[:-1])


############################################################
# IMAGE TRANSFORMS
############################################################

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


############################################################
# HELPER FUNCTIONS
############################################################

def load_image(path):
    return transform(Image.open(path).convert("RGB"))


def get_activation(img_tensor):
    """
    Extract 2048-d penultimate layer activation
    and min-max normalize to [0,1].
    """
    with torch.no_grad():

        feat = model(img_tensor).view(-1)

        feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)

        return feat.cpu().numpy()


############################################################
# LOAD INDIA / TURKEY IMAGE PATHS
############################################################

india_folder = os.path.join(base_path, "India")
turkey_folder = os.path.join(base_path, "Turkey")

india_images = sorted([
    os.path.join(india_folder, f)
    for f in os.listdir(india_folder)
    if f.lower().endswith(image_extensions)
])

turkey_images = sorted([
    os.path.join(turkey_folder, f)
    for f in os.listdir(turkey_folder)
    if f.lower().endswith(image_extensions)
])

assert len(india_images) >= 200, "Need at least 200 India images"
assert len(turkey_images) >= 100, "Need at least 100 Turkey images"


############################################################
# STEP 1 — BUILD INDIA BASELINE HISTOGRAM
############################################################

baseline_feats = []

print("Building India baseline histogram...")

for img_path in india_images[:num_images_avg]:

    img = load_image(img_path).unsqueeze(0).to(device)

    feat = get_activation(img)

    baseline_feats.append(feat)

baseline_feats = np.stack(baseline_feats)

# Average activations across 100 India images
india_baseline_hist = baseline_feats.mean(axis=0)

# Normalize to probability distribution
india_baseline_hist = india_baseline_hist / (india_baseline_hist.sum() + 1e-8)


############################################################
# PLOT INDIA BASELINE HISTOGRAM
############################################################

plt.figure(figsize=(12,5))
plt.plot(india_baseline_hist)
plt.title("India Baseline Average Activation Histogram")
plt.xlabel("Feature Dimension")
plt.ylabel("Normalized Activation")
plt.grid()
plt.show()


############################################################
# STEP 2 — INDIA TEST KL
############################################################

india_kl_values = []

print("Computing India test KL divergences...")

for img_path in india_images[num_images_avg:num_images_avg+num_images_test]:

    img = load_image(img_path).unsqueeze(0).to(device)

    feat = get_activation(img)

    feat = feat / (feat.sum() + 1e-8)

    kl = entropy(feat + 1e-8, india_baseline_hist + 1e-8)

    india_kl_values.append(kl)

india_avg_kl = np.mean(india_kl_values)


############################################################
# STEP 3 — TURKEY KL
############################################################

turkey_kl_values = []

print("Computing Turkey KL divergences...")

for img_path in turkey_images[:num_images_test]:

    img = load_image(img_path).unsqueeze(0).to(device)

    feat = get_activation(img)

    feat = feat / (feat.sum() + 1e-8)

    kl = entropy(feat + 1e-8, india_baseline_hist + 1e-8)

    turkey_kl_values.append(kl)

turkey_avg_kl = np.mean(turkey_kl_values)


############################################################
# PRINT RESULTS
############################################################

print("\n==============================")
print("FINAL RESULTS")
print("==============================")

print(f"India Test Avg KL vs India Baseline: {india_avg_kl:.6f}")
print(f"Turkey Avg KL vs India Baseline:     {turkey_avg_kl:.6f}")


############################################################
# PLOT KL DISTRIBUTIONS
############################################################

plt.figure(figsize=(10,5))

plt.hist(india_kl_values,  bins=20, alpha=0.6, label="India Test", color='#3b0f70')
plt.hist(turkey_kl_values, bins=20, alpha=0.6, label="Turkey",     color='darkorange')

plt.axvline(india_avg_kl,  color='#3b0f70', linestyle='--', linewidth=1.5)
plt.axvline(turkey_avg_kl, color='darkorange',   linestyle='--', linewidth=1.5)

plt.text(india_avg_kl,  plt.ylim()[1] * 0.92, f'India avg: {india_avg_kl:.4f}',
         color='#3b0f70', ha='center', fontsize=10, fontweight='bold')
plt.text(turkey_avg_kl, plt.ylim()[1] * 0.82, f'Turkey avg: {turkey_avg_kl:.4f}',
         color='darkorange',   ha='center', fontsize=10, fontweight='bold')

plt.legend()
plt.title("ResNet-50 — KL Divergence Distribution")
plt.xlabel("KL Divergence")
plt.ylabel("Frequency")
plt.grid()
plt.show()


### MobileNetV2 — India Baseline and KL Distributions

In [ ]:
############################################################
# LOAD MOBILENETV2
############################################################
import os
import torch
import torch.nn as nn
from torchvision import models, transforms
from PIL import Image
import numpy as np
from scipy.stats import entropy
import matplotlib.pyplot as plt
mobilenet = models.mobilenet_v2(weights=models.MobileNet_V2_Weights.DEFAULT).to(device)
mobilenet.eval()

# Remove final classifier → penultimate activations (1280-d)
mobilenet_feat_model = nn.Sequential(*list(mobilenet.children())[:-1])

############################################################
# MOBILENET HELPER
############################################################

def get_activation_mobilenet(img_tensor):
    """
    Extract 1280-d penultimate layer activation
    and min-max normalize to [0,1].
    """
    with torch.no_grad():
        feat = mobilenet_feat_model(img_tensor)
        feat = feat.mean([2, 3]).view(-1)          # global avg pool → (1280,)
        feat = (feat - feat.min()) / (feat.max() - feat.min() + 1e-8)
        return feat.cpu().numpy()

############################################################
# STEP 1 — MOBILENET INDIA BASELINE
############################################################

mobilenet_baseline_feats = []

print("Building MobileNetV2 India baseline histogram...")

for img_path in india_images[:num_images_avg]:
    img = load_image(img_path).unsqueeze(0).to(device)
    feat = get_activation_mobilenet(img)
    mobilenet_baseline_feats.append(feat)

mobilenet_baseline_feats = np.stack(mobilenet_baseline_feats)

mobilenet_india_baseline = mobilenet_baseline_feats.mean(axis=0)
mobilenet_india_baseline = mobilenet_india_baseline / (mobilenet_india_baseline.sum() + 1e-8)

############################################################
# STEP 2 — MOBILENET INDIA TEST KL
############################################################

mobilenet_india_kl_values = []

print("Computing MobileNetV2 India test KL divergences...")

for img_path in india_images[num_images_avg:num_images_avg + num_images_test]:
    img = load_image(img_path).unsqueeze(0).to(device)
    feat = get_activation_mobilenet(img)
    feat = feat / (feat.sum() + 1e-8)
    kl = entropy(feat + 1e-8, mobilenet_india_baseline + 1e-8)
    mobilenet_india_kl_values.append(kl)

mobilenet_india_avg_kl = np.mean(mobilenet_india_kl_values)

############################################################
# STEP 3 — MOBILENET TURKEY KL
############################################################

mobilenet_turkey_kl_values = []

print("Computing MobileNetV2 Turkey KL divergences...")

for img_path in turkey_images[:num_images_test]:
    img = load_image(img_path).unsqueeze(0).to(device)
    feat = get_activation_mobilenet(img)
    feat = feat / (feat.sum() + 1e-8)
    kl = entropy(feat + 1e-8, mobilenet_india_baseline + 1e-8)
    mobilenet_turkey_kl_values.append(kl)

mobilenet_turkey_avg_kl = np.mean(mobilenet_turkey_kl_values)

############################################################
# PRINT RESULTS
############################################################

print("\n==============================")
print("MOBILENETV2 FINAL RESULTS")
print("==============================")
print(f"India Test Avg KL vs India Baseline: {mobilenet_india_avg_kl:.6f}")
print(f"Turkey Avg KL vs India Baseline:     {mobilenet_turkey_avg_kl:.6f}")

############################################################
# PLOT KL DISTRIBUTIONS
############################################################

plt.figure(figsize=(10,5))

plt.hist(mobilenet_india_kl_values,  bins=20, alpha=0.6, label="India Test", color='#3b0f70')
plt.hist(mobilenet_turkey_kl_values, bins=20, alpha=0.6, label="Turkey",     color='darkorange')

plt.axvline(mobilenet_india_avg_kl,  color='#3b0f70', linestyle='--', linewidth=1.5)
plt.axvline(mobilenet_turkey_avg_kl, color='darkorange',   linestyle='--', linewidth=1.5)

plt.text(mobilenet_india_avg_kl,  plt.ylim()[1] * 0.92, f'India avg: {mobilenet_india_avg_kl:.4f}',
         color='#3b0f70', ha='center', fontsize=10, fontweight='bold')
plt.text(mobilenet_turkey_avg_kl, plt.ylim()[1] * 0.82, f'Turkey avg: {mobilenet_turkey_avg_kl:.4f}',
         color='darkorange',   ha='center', fontsize=10, fontweight='bold')

plt.legend()
plt.title("MobileNetV2 — KL Divergence Distribution")
plt.xlabel("KL Divergence")
plt.ylabel("Frequency")
plt.grid()
plt.show()
